# 00. Graph Data Overview

This folder is reserved for graph-only experiments.

Current scope:

- `GCN`
- `GraphSAGE`
- `GAT`
- `GGNN`

Each model notebook uses the same `10,000`-node / `274,561`-edge sampled graph and the same split logic, so the comparisons stay fair.
By default, these notebooks run in a stricter setup with graph-stat columns disabled.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import seaborn as sns

sys.path.append(str(Path.cwd()))

from graph_model_utils import GRAPH_STAT_COLS, get_feature_groups, load_frames

sns.set_theme(style="whitegrid")
nodes_df, edges_df = load_frames()


In [ ]:
print("nodes_df.shape =", nodes_df.shape)
print("edges_df.shape =", edges_df.shape)
display(nodes_df.head())
display(edges_df.head())


In [ ]:
feature_groups = get_feature_groups(nodes_df)
pd.Series({name: len(cols) for name, cols in feature_groups.items()} | {"graph_stats": len(GRAPH_STAT_COLS)}).sort_values(ascending=False)


In [ ]:
label_summary = pd.DataFrame(
    {
        "count": nodes_df["label"].value_counts().sort_index(),
    }
)
label_summary["ratio"] = label_summary["count"] / label_summary["count"].sum()
display(label_summary)

plt.figure(figsize=(5, 4))
sns.countplot(data=nodes_df, x="label")
plt.title("Graph Sample Label Distribution")
plt.show()


In [ ]:
nx_graph = nx.from_pandas_edgelist(
    edges_df,
    source="src_node_id",
    target="dst_node_id",
    create_using=nx.DiGraph(),
)

connected_nodes = set(edges_df["src_node_id"]).union(edges_df["dst_node_id"])
summary = {
    "num_nodes_csv": len(nodes_df),
    "num_edges_csv": len(edges_df),
    "graph_nodes": nx_graph.number_of_nodes(),
    "graph_edges": nx_graph.number_of_edges(),
    "isolated_nodes": int((~nodes_df["node_id"].isin(connected_nodes)).sum()),
    "density": nx.density(nx_graph),
    "self_loops": nx.number_of_selfloops(nx_graph),
    "weak_components": nx.number_weakly_connected_components(nx_graph),
}
pd.Series(summary)


In [ ]:
component_sizes = sorted((len(c) for c in nx.weakly_connected_components(nx_graph)), reverse=True)
component_df = pd.DataFrame({"component_size": component_sizes})
display(component_df.head(10))

plt.figure(figsize=(7, 4))
sns.histplot(component_df["component_size"], bins=30)
plt.title("Weakly Connected Component Size Distribution")
plt.show()


In [ ]:
degree_cols = [
    "full_in_degree",
    "full_out_degree",
    "full_total_degree",
    "sub_in_degree",
    "sub_out_degree",
    "sub_total_degree",
]

display(nodes_df[degree_cols + ["label"]].describe().T)

plt.figure(figsize=(8, 5))
sns.boxplot(data=nodes_df[["sub_total_degree", "label"]], x="label", y="sub_total_degree")
plt.yscale("log")
plt.title("Subgraph Degree By Label")
plt.show()


## Next

Run the model notebooks in this folder:

- `01_gcn.ipynb`
- `02_graphsage.ipynb`
- `03_gat.ipynb`
- `04_ggnn.ipynb`
- `05_model_comparison.ipynb`
